# Feature Extraction

In this notebook we will extract the MFCC, LFCC, and CQCC features from the preprocessed data.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd()
MANIFESTS_DIR = PROJECT_ROOT / "manifests"
WAV16_DIR = PROJECT_ROOT / "data" / "wav16"
FEATURES_DIR = PROJECT_ROOT / "features"
FEATURES_DIR.mkdir(exist_ok=True, parents=True)

# Which preprocessed manifests to run on:
TRAIN_PRE = MANIFESTS_DIR / "LA_train_preprocessed.csv"
DEV_PRE   = MANIFESTS_DIR / "LA_dev_preprocessed.csv"
EVAL_PRE  = MANIFESTS_DIR / "LA_eval_preprocessed.csv"

# Frame params (standard for speech)
SR = 16000
N_FFT = 512           # window for STFT (~32ms); keep consistent with literature choices
HOP_LENGTH = 160      # 10 ms hop @16k
WIN_LENGTH = 400      # 25 ms window @16k

# Cepstral dims
MFCC_N_MELS = 40
MFCC_N_CEP = 20

LFCC_N_FILTERS = 40   # number of linear filterbanks
LFCC_N_CEP = 20

CQCC_N_CEP = 20       # how many cepstral coefficients to keep after DCT on log-CQT

# Pooling function (mean + std)
def pool_mean_std(feat: np.ndarray):
    # feat: (T, D) => return (2*D,)
    mean = np.mean(feat, axis=0)
    std  = np.std(feat, axis=0)
    return np.concatenate([mean, std], axis=0)

print("Setup complete. SR=", SR, "hop=", HOP_LENGTH, "win=", WIN_LENGTH)

Setup complete. SR= 16000 hop= 160 win= 400


## MFCC

In [ ]:
import librosa
import os
import json

def extract_mfcc_from_wav(wav_path, sr=SR, n_mfcc=MFCC_N_CEP, n_mels=MFCC_N_MELS,
                          n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # compute mfcc: shape (n_mfcc, T)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_mels=n_mels,
                                n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    # transpose to (T, D)
    return mfcc.T.astype(np.float32)

def run_mfcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"MFCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        label = int(r['label'])
        try:
            frames = extract_mfcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    # write a small manifest for these features
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote MFCC features to", out_dir, "and manifest", out_manifest)
    return out_manifest

MFCC LA_train_preprocessed.csv: 100%|██████████| 319/319 [00:24<00:00, 13.19it/s]

Wrote MFCC features to /scratch1/kodachi/project/manifests/mfcc and manifest /scratch1/kodachi/project/manifests/mfcc_LA_train_preprocessed_feat_manifest.csv


In [ ]:

# Example (run per split; takes a while for all files)
mfcc_train_manifest = run_mfcc_on_manifest(TRAIN_PRE, MANIFESTS_DIR / "mfcc", pooled=True, save_frames=False)

In [4]:
mfcc_dev_manifest = run_mfcc_on_manifest(DEV_PRE, MANIFESTS_DIR / "mfcc", pooled=True, save_frames=False)

MFCC LA_dev_preprocessed.csv: 100%|██████████| 24844/24844 [31:00<00:00, 13.36it/s] 

Wrote MFCC features to /scratch1/kodachi/project/manifests/mfcc and manifest /scratch1/kodachi/project/manifests/mfcc_LA_dev_preprocessed_feat_manifest.csv


In [ ]:
mfcc_dev_manifest = run_mfcc_on_manifest(EVAL_PRE, MANIFESTS_DIR / "mfcc", pooled=True, save_frames=False)

## LFCC

In [ ]:
import scipy.fftpack as fftpack

def linear_filterbank(n_filters=LFCC_N_FILTERS, n_fft=N_FFT, sr=SR):
    # Build center frequencies linearly spaced from 0..sr/2
    # Create triangular filters across FFT bins.
    freqs = np.linspace(0, sr/2, int(n_fft//2 + 1))
    # choose filter center frequencies equally spaced
    centers = np.linspace(0, sr/2, n_filters + 2)  # include edges
    fb = np.zeros((n_filters, len(freqs)), dtype=np.float32)
    for i in range(n_filters):
        left = centers[i]
        center = centers[i+1]
        right = centers[i+2]
        # triangle
        up_slope = (freqs - left) / (center - left + 1e-9)
        down_slope = (right - freqs) / (right - center + 1e-9)
        tri = np.maximum(0, np.minimum(up_slope, down_slope))
        fb[i, :] = tri
    return fb, freqs

def extract_lfcc_from_wav(wav_path, sr=SR, n_filters=LFCC_N_FILTERS, n_cep=LFCC_N_CEP,
                          n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH):
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # compute power spectrogram (shape: (freq_bins, T))
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length, center=True))**2
    fb, freqs = linear_filterbank(n_filters=n_filters, n_fft=n_fft, sr=sr)
    # map S bins to fb via interpolation: S has freq bins corresponding to FFT bins
    # fft_freqs:
    fft_freqs = np.linspace(0, sr/2, S.shape[0])
    # interpolate filterbank to match S.shape[0] if needed (our fb already matches)
    # apply filterbank: filterbank @ S  -> (n_filters, T)
    S_power = S[:fb.shape[1], :] if S.shape[0] >= fb.shape[1] else S
    # If shapes mismatch, resample fb to S.shape[0]
    if fb.shape[1] != S.shape[0]:
        # interpolate fb over fft_freqs
        fb_interp = np.zeros((fb.shape[0], S.shape[0]), dtype=np.float32)
        fb_freqs = freqs
        target_freqs = fft_freqs
        for i in range(fb.shape[0]):
            fb_interp[i, :] = np.interp(target_freqs, fb_freqs, fb[i, :])
        fb_use = fb_interp
    else:
        fb_use = fb
    filt_energies = np.dot(fb_use, S)
    # avoid zeros before log
    filt_energies[filt_energies <= 1e-12] = 1e-12
    logE = np.log(filt_energies)
    # DCT on log energies along filter axis -> cepstral coefficients
    cep = fftpack.dct(logE, axis=0, type=2, norm='ortho')[:n_cep, :]
    # transpose to (T, D)
    return cep.T.astype(np.float32)

def run_lfcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"LFCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        label = int(r['label'])
        try:
            frames = extract_lfcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote LFCC features to", out_dir, "and manifest", out_manifest)
    return out_manifest

LFCC LA_train_preprocessed.csv: 100%|██████████| 319/319 [00:10<00:00, 30.90it/s]

Wrote LFCC features to /scratch1/kodachi/project/manifests/lfcc and manifest /scratch1/kodachi/project/manifests/lfcc_LA_train_preprocessed_feat_manifest.csv


In [ ]:

# Example
lfcc_train_manifest = run_lfcc_on_manifest(TRAIN_PRE, MANIFESTS_DIR / "lfcc", pooled=True, save_frames=False)

In [ ]:
lfcc_dev_manifest = run_lfcc_on_manifest(DEV_PRE, MANIFESTS_DIR / "lfcc", pooled=True, save_frames=False)

In [ ]:
lfcc_eval_manifest = run_lfcc_on_manifest(EVAL_PRE, MANIFESTS_DIR / "lfcc", pooled=True, save_frames=False)

## CQCC

In [ ]:
def extract_cqcc_from_wav(wav_path, sr=SR, n_cq_bins=84, hop_length=HOP_LENGTH, n_cep=CQCC_N_CEP):
    """
    Simple CQCC-like extractor:
      - compute CQT magnitude (log)
      - treat log-CQT like spectral representation, apply DCT across freq bins
      - keep first n_cep coefficients per frame
    n_cq_bins: number of CQT bins (recommend ~84 or 96)
    """
    y, sr0 = librosa.load(wav_path, sr=sr, mono=True)
    # CQT: returns (freq_bins, frames)
    C = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=n_cq_bins, bins_per_octave=12)
    C_mag = np.abs(C)
    # convert to power or magnitude, avoid zeros
    C_mag[C_mag <= 1e-12] = 1e-12
    logC = np.log(C_mag)
    # Now apply DCT across frequency axis (axis=0) to get cepstral-like coefficients
    # logC: (freq_bins, T)
    cep = fftpack.dct(logC, axis=0, type=2, norm='ortho')[:n_cep, :]
    return cep.T.astype(np.float32)

def run_cqcc_on_manifest(manifest_csv, out_dir: Path, pooled=True, save_frames=False):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(manifest_csv)
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f"CQCC {manifest_csv.name}"):
        utt = r['utt']
        wav = r['wav']
        label = int(r['label'])
        try:
            frames = extract_cqcc_from_wav(wav)
            if save_frames:
                np.save(out_dir / f"{utt}_frames.npy", frames)
            vec = pool_mean_std(frames) if pooled else frames
            np.save(out_dir / f"{utt}.npy", vec)
            rows.append((utt, str((out_dir / f'{utt}.npy').resolve()), label))
        except Exception as e:
            print(f"Failed {utt}: {e}")
    out_manifest = out_dir.parent / f"{out_dir.name}_{manifest_csv.stem}_feat_manifest.csv"
    pd.DataFrame(rows, columns=["utt","feat_path","label"]).to_csv(out_manifest, index=False)
    print("Wrote CQCC-like features to", out_dir, "and manifest", out_manifest)
    return out_manifest

CQCC LA_train_preprocessed.csv:   9%|▉         | 30/319 [00:01<00:17, 16.48it/s]/home1/kodachi/.conda/envs/ee599/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=505
  warnings.warn(
CQCC LA_train_preprocessed.csv:  70%|██████▉   | 223/319 [00:13<00:05, 16.61it/s]/home1/kodachi/.conda/envs/ee599/lib/python3.12/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=512 is too large for input signal of length=483
  warnings.warn(
CQCC LA_train_preprocessed.csv: 100%|██████████| 319/319 [00:19<00:00, 16.67it/s]

Wrote CQCC-like features to /scratch1/kodachi/project/manifests/cqcc and manifest /scratch1/kodachi/project/manifests/cqcc_LA_train_preprocessed_feat_manifest.csv


In [ ]:

# Example
cqcc_train_manifest = run_cqcc_on_manifest(TRAIN_PRE, MANIFESTS_DIR / "cqcc", pooled=True, save_frames=False)

In [ ]:
cqcc_dev_manifest = run_cqcc_on_manifest(DEV_PRE, MANIFESTS_DIR / "cqcc", pooled=True, save_frames=False)

In [ ]:
cqcc_eval_manifest = run_cqcc_on_manifest(TRAIN_PRE, MANIFESTS_DIR / "cqcc", pooled=True, save_frames=False)